# 01 — DenseNet121: training, evaluation and sample prediction

**Member 3 — DenseNet + Backend.** Trains DenseNet121 on APTOS 2019 following
`docs/common_experiment_settings.md`, evaluates it on the test split and saves everything to Google Drive.

**Before running:**
1. **Runtime → Change runtime type → T4 GPU** → Save.
2. Use the **same Google account** that holds `MyDrive/DR_project/data/aptos_train.zip`.
3. Then **Runtime → Run all**. Total time ≈ 30–45 min.

All code lives in `src/` of the GitHub repo — this notebook only calls it, so the same code
is used here, in the tests and in the backend.

Outputs (saved to `MyDrive/DR_project/densenet121/`):
| File | What |
|---|---|
| `densenet121_dr.keras` | trained model (needed by the backend) |
| `densenet121_dr_info.json` | model metadata (used by `GET /model-info`) |
| `densenet121_history.json` | loss/accuracy per epoch |
| `results/densenet121_metrics.json` | test metrics |
| `results/*.png` | confusion matrix, training curves, samples |
| `splits/*.csv` | the train/val/test split used |

## 1. Check the GPU

In [ ]:
import tensorflow as tf, keras
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus)
assert gpus, 'No GPU! Runtime → Change runtime type → T4 GPU, then run again.'

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/DR_project'
DRIVE_ZIP  = f'{DRIVE_ROOT}/data/aptos_train.zip'
DRIVE_OUT  = f'{DRIVE_ROOT}/densenet121'
SPLITS_DIR = f'{DRIVE_ROOT}/splits'          # kept in Drive so every run uses the same split
DATA_DIR   = '/content/aptos'                # fast local disk
OUT_DIR    = '/content/output'

assert os.path.exists(DRIVE_ZIP), f'Dataset not found: {DRIVE_ZIP} — wrong Google account?'
print('✅ Dataset zip found')

## 3. Get the project code from GitHub

In [ ]:
!rm -rf /content/repo
!git clone -q https://github.com/ShaikNousheen1700/diabetic-retinopathy-member3.git /content/repo
import sys
sys.path.insert(0, '/content/repo')
!cd /content/repo && git log --oneline -1

## 4. Copy the dataset to local disk and unzip (≈ 2 min)

In [ ]:
if not os.path.exists(f'{DATA_DIR}/train.csv'):
    !cp "{DRIVE_ZIP}" /content/aptos_train.zip
    !unzip -q -o /content/aptos_train.zip -d {DATA_DIR}
    !rm /content/aptos_train.zip

import pandas as pd
df = pd.read_csv(f'{DATA_DIR}/train.csv')
n_images = len([f for f in os.listdir(f'{DATA_DIR}/train_images') if f.endswith('.png')])
print('CSV rows:', len(df), '| PNG images:', n_images)
assert len(df) == n_images == 3662
print('✅ Dataset ready')

## 5. Explore the data
Class distribution (APTOS is imbalanced) and one example per class: original vs preprocessed
(crop black border → resize 224×224 — exactly what the model receives).

In [ ]:
import matplotlib.pyplot as plt
from src import config
from src.preprocessing import load_image, preprocess_image

RESULTS_DIR = f'{OUT_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

counts = df['diagnosis'].value_counts().sort_index()
print(pd.DataFrame({'class': config.CLASS_NAMES, 'images': counts.values,
                    'percent': (100 * counts / len(df)).round(1).values}).to_string(index=False))

plt.figure(figsize=(7, 4))
plt.bar(config.CLASS_NAMES, counts.values, color='steelblue')
for i, c in enumerate(counts.values):
    plt.text(i, c + 20, str(c), ha='center')
plt.title('APTOS 2019 — images per class'); plt.ylabel('Images')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/class_distribution.png', dpi=150); plt.show()

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for c in range(5):
    id_code = df[df['diagnosis'] == c]['id_code'].iloc[0]
    raw = load_image(f'{DATA_DIR}/train_images/{id_code}.png')
    axes[0, c].imshow(raw); axes[0, c].set_title(f'{c}: {config.CLASS_NAMES[c]}\n{raw.shape[1]}x{raw.shape[0]}')
    axes[1, c].imshow(preprocess_image(raw)); axes[1, c].set_title('preprocessed 224x224')
for ax in axes.ravel(): ax.axis('off')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/sample_images.png', dpi=120); plt.show()

## 6. Train (≈ 25–40 min)
`run()` in `src/train.py` does: split (stratified 70/15/15, seed 42) → preprocess all images →
**phase 1** (head only, 10 epochs, lr 1e-3) → **phase 2** (fine-tune last dense block, up to 20 epochs,
lr 1e-5, early stopping). The best model by validation loss is saved.

If your team shares split files, put them in `MyDrive/DR_project/splits/` before running — they will be used instead.

In [ ]:
from src.train import run
result = run(DATA_DIR, OUT_DIR, splits_dir=SPLITS_DIR, weights='imagenet')

## 7. Evaluate on the test split

In [ ]:
from src.evaluate import evaluate_model, save_results, print_summary
from IPython.display import Image, display

model_path = f'{OUT_DIR}/{config.MODEL_FILENAME}'
metrics, y_pred, probs = evaluate_model(result['model'], result['X_test'], result['y_test'], model_path)
metrics['epochs_trained'] = result['info']['epochs_trained']
metrics['train_minutes'] = round(result['info']['train_seconds'] / 60, 1)
paths = save_results(metrics, result['history'], RESULTS_DIR)
print_summary(metrics)
for p in paths[1:]:
    display(Image(p))

## 8. Sample predictions
Uses `DRPredictor` from `src/predict.py` — the same class the backend uses. It loads the **saved file**, proving the file works.

In [ ]:
from src.predict import DRPredictor
predictor = DRPredictor(model_path)

test_df = result['test_df']
samples = test_df.groupby('diagnosis').head(1).sort_values('diagnosis')   # one per class
fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4.5))
for ax, (_, row) in zip(axes, samples.iterrows()):
    path = f"{DATA_DIR}/train_images/{row['id_code']}.png"
    pred = predictor.predict_path(path)
    ok = pred['class_id'] == row['diagnosis']
    ax.imshow(preprocess_image(load_image(path))); ax.axis('off')
    ax.set_title(f"true: {config.CLASS_NAMES[row['diagnosis']]}\npred: {pred['class_name']} "
                 f"({pred['confidence']:.0%})", color='green' if ok else 'red')
    print(row['id_code'], pred)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/sample_predictions.png', dpi=120); plt.show()

## 9. Save everything to Google Drive

In [ ]:
!mkdir -p "{DRIVE_OUT}"
!cp -r {OUT_DIR}/* "{DRIVE_OUT}/"
!ls -lh "{DRIVE_OUT}" "{DRIVE_OUT}/results"
print('\n✅ Saved to', DRIVE_OUT)
print('Next: download densenet121_dr.keras + densenet121_dr_info.json into the project models/ folder,')
print('and the results/ files into the project results/ folder.')